# Scanning a malicious-style pickle with the real `picklescan` tool

**Goal:** scan model files for dangerous pickle imports without loading them.
**Runtime:** Colab CPU (any). **Time:** 15 minutes.
**Offline twin:** `../labs/picklescan/` (same idea, with tests, no install needed).

**Not executed by the repo author: run in Colab or in your course lab; versions may need updates.**

**Safety:** the sample file below is hand-written bytes whose "command" is `echo CANARY`. We only SCAN it.
We never call `pickle.load()` on it. Do not load untrusted files in a notebook that holds real credentials.

## 1. Install
The pin below is the version the author expects to exist. If pip cannot find it, install without the pin and note the version you got.

In [ ]:
!pip install -q picklescan==0.0.21
!pip show picklescan | head -3

## 2. Create harmless malicious-style samples
`c os\nsystem` is the textbook shape of a dangerous pickle: import `os.system`, then call it. Here it only echoes a word.

In [ ]:
import pickle, pathlib
d = pathlib.Path("samples"); d.mkdir(exist_ok=True)
(d / "clean.pkl").write_bytes(pickle.dumps({"layer1": [0.1, 0.2]}))
(d / "evil_style.pkl").write_bytes(b"cos\nsystem\n(S'echo CANARY'\ntR.")
print(sorted(p.name for p in d.iterdir()))

## 3. Scan the folder with the CLI
The CLI prints the files it scanned and any dangerous globals it found. Run `picklescan --help` if a flag name differs in your version (the common form is `--path`).

In [ ]:
!picklescan --help | head -20
!picklescan --path samples

**Read the output.** You should see the dangerous global (for example `posix.system` or `os.system`) reported for `evil_style.pkl` and a clean result for `clean.pkl`. Exact wording depends on the version.

## 4. Use it from Python (optional)
The module also has a Python API. Names may change between versions, so check `help(picklescan.scanner)` first.

In [ ]:
import picklescan.scanner as ps
print([n for n in dir(ps) if "scan" in n.lower()])
# Typical call (verify in your version): result = ps.scan_file_path("samples/evil_style.pkl")
# print(result.infected_files, result.scanned_files)

## 5. Think about it
1. The tool decodes the file; it does not run it. Why is that the whole point?
2. A scanner reduces risk, it does not remove it. What is the safer format for weights? (`safetensors`: data only.)
3. Where in a CI/CD pipeline would you add this scan? Before the model is registered, and again before deploy.
4. Try the evasions from `../labs/picklescan/samples.py` (memo trick, broken stream). Does the real tool catch them?